In [ ]:
from data_engineering.storage.connection import create_session, create_client
from data_engineering.storage.s3 import read_json
from data_engineering.storage.query import query_parquet

session = create_session()
client = create_client(session)

manifest = read_json(
    client,
    "gold/keejob/bfce5dcc9bf046eaaa8aff9070e0a269/manifest.json",
)


In [3]:
manifest

{'run_id': 'd277c71f12bc4c5abb498285856a0ed0',
 'status': 'complete',
 'bucket': 'tunisian-jobs',
 'input_silver_key': 'silver/keejob/bfce5dcc9bf046eaaa8aff9070e0a269.parquet',
 'input_count': 679,
 'started_at': '2026-10-20T09:49:45.281152+00:00',
 'completed_at': '2026-10-20T09:49:45.495447+00:00',
 'company_ranking_excluded_count': 114,
 'datasets': {'company_posting_counts': {'key': 'gold/keejob/bfce5dcc9bf046eaaa8aff9070e0a269/runs/d277c71f12bc4c5abb498285856a0ed0/company_posting_counts.parquet',
   'row_count': 252},
  'governorate_posting_counts': {'key': 'gold/keejob/bfce5dcc9bf046eaaa8aff9070e0a269/runs/d277c71f12bc4c5abb498285856a0ed0/governorate_posting_counts.parquet',
   'row_count': 20},
  'salary_summary': {'key': 'gold/keejob/bfce5dcc9bf046eaaa8aff9070e0a269/runs/d277c71f12bc4c5abb498285856a0ed0/salary_summary.parquet',
   'row_count': 1},
  'salary_postings': {'key': 'gold/keejob/bfce5dcc9bf046eaaa8aff9070e0a269/runs/d277c71f12bc4c5abb498285856a0ed0/salary_postings.par

In [4]:
company_key = manifest["datasets"]["company_posting_counts"]["key"]

In [5]:
result = query_parquet(
    session,
    company_key,
    """
    SELECT company_name, posting_count
    FROM dataset
    ORDER BY posting_count DESC, company_name ASC
    LIMIT 10
    """,
)

result

,company_name,posting_count
0,OUTSOURCIA,16
1,POULINA GROUP HOLDING,16
2,TUNTEX,15
3,SOQUIBAT,14
4,Talent Match,14
5,EL AMEL MICROFINANCE,12
6,HR HOUSE INTERNATIONAL,11
7,TECNOCASA GROUP,11
8,ACTIVE CONTACT,9
9,BEN YAGHLANE SHOPS,9


In [ ]:
governorate_key = manifest["datasets"]["governorate_posting_counts"]["key"]

governorate_result = query_parquet(
    session,
    governorate_key,
    """
    SELECT
        source,
        governorate,
        posting_count,
        ROUND(
            100.0 * posting_count
            / SUM(posting_count) OVER (PARTITION BY source),
            2
        ) AS posting_share_percent
    FROM dataset
    ORDER BY posting_count DESC, governorate ASC
    """,
)

governorate_result

,source,governorate,posting_count,posting_share_percent
0,keejob,Tunis,221,32.55
1,keejob,Ariana,105,15.46
2,keejob,Ben Arous,84,12.37
3,keejob,Sousse,49,7.22
4,keejob,Unknown,47,6.92
5,keejob,Nabeul,37,5.45
6,keejob,Zaghouan,35,5.15
7,keejob,La Manouba,20,2.95
8,keejob,Sfax,18,2.65
9,keejob,Monastir,16,2.36


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/amine/Projects/data engineering/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
  File "/home/amine/Projects/data engineering/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 372, in dispatch_control
    await self.process_control(msg)
  File "/home/amine/Projects/data engineering/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 378, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/amine/Projects/data engineering/.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/amine/